In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample

In [2]:
try:
    import shap
except ImportError as e:
    raise ImportError(
        "This script requires the 'shap' package (already a dependency of "
        "XAI.ipynb / RandomForest.ipynb in the original repository). "
        "Install with: pip install shap"
    ) from e

RANDOM_STATE = 42
DATA_PATH = "online_shoppers_intention.csv"
OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)

N_BOOTSTRAPS = 30
EVAL_SAMPLE_SIZE = 500          # fixed evaluation sample SHAP is computed on every time
CORR_THRESHOLD = 0.7            # |Spearman rho| above which two features are "structurally correlated"
FOCUS_FEATURE = "Month_Nov"

c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_data(path=DATA_PATH):
    df = pd.read_csv(path)
    NOMINAL_INT_COLS = ["OperatingSystems", "Browser", "Region", "TrafficType"]
    df[NOMINAL_INT_COLS] = df[NOMINAL_INT_COLS].astype("category")
    df_encoded = pd.get_dummies(df, drop_first=True, dtype=int)
    X = df_encoded.drop(columns=["Revenue"])
    y = df_encoded["Revenue"]
    return X, y

In [4]:
def positive_class_shap(shap_output):
    """
    Normalize TreeExplainer output to a single (n_samples, n_features) array
    of SHAP values for the positive ("Purchase") class, across SHAP versions
    that return either a 3D ndarray (n_samples, n_features, n_classes), a
    list of per-class 2D arrays, or already a single 2D array.
    """
    if isinstance(shap_output, list):
        # Older SHAP: list of arrays, one per class
        return shap_output[1] if len(shap_output) > 1 else shap_output[0]
    arr = np.asarray(shap_output)
    if arr.ndim == 3:
        return arr[:, :, 1]
    return arr

In [5]:
def cluster_correlated_features(X, threshold=CORR_THRESHOLD):
    corr = X.corr(method="spearman").abs()
    dist = 1 - corr
    # Guard against tiny numerical negatives / asymmetries before squareform
    dist_vals = dist.values
    dist_vals = (dist_vals + dist_vals.T) / 2
    np.fill_diagonal(dist_vals, 0.0)
    condensed = squareform(dist_vals, checks=False)
    Z = linkage(condensed, method="average")
    cluster_labels = fcluster(Z, t=1 - threshold, criterion="distance")

    cluster_df = pd.DataFrame({"Feature": X.columns, "Cluster": cluster_labels})
    cluster_df = cluster_df.sort_values("Cluster")
    cluster_df.to_csv(os.path.join(OUT_DIR, "feature_correlation_clusters.csv"), index=False)

    multi_member_clusters = cluster_df.groupby("Cluster").filter(lambda g: len(g) > 1)
    print("Correlated feature clusters (|Spearman rho| >= %.2f):" % threshold)
    for cid, group in multi_member_clusters.groupby("Cluster"):
        print(f"  Cluster {cid}: {list(group.Feature.values)}")

    return cluster_df

In [6]:
def bootstrap_shap_stability(X_train_s, y_train, X_eval_s, feature_names, n_boot=N_BOOTSTRAPS):
    rng = np.random.RandomState(RANDOM_STATE)

    # Reference fit (bootstrap index 0 = "original", no resampling)
    ref_rf = RandomForestClassifier(
        n_estimators=300, min_samples_leaf=3, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    )
    ref_rf.fit(X_train_s, y_train)
    ref_explainer = shap.TreeExplainer(ref_rf, feature_perturbation="tree_path_dependent")
    ref_shap = positive_class_shap(ref_explainer.shap_values(X_eval_s, check_additivity=False))
    ref_importance = np.abs(ref_shap).mean(axis=0)
    ref_rank = pd.Series(ref_importance, index=feature_names).rank(ascending=False)

    boot_importances = []
    boot_rank_corrs = []
    focus_records = []

    for b in range(n_boot):
        seed = RANDOM_STATE + b + 1
        X_res, y_res = resample(X_train_s, y_train, replace=True, random_state=seed)

        rf_b = RandomForestClassifier(
            n_estimators=300, min_samples_leaf=3, class_weight="balanced",
            random_state=seed, n_jobs=-1
        )
        rf_b.fit(X_res, y_res)

        explainer_b = shap.TreeExplainer(rf_b, feature_perturbation="tree_path_dependent")
        shap_b = positive_class_shap(explainer_b.shap_values(X_eval_s, check_additivity=False))

        importance_b = np.abs(shap_b).mean(axis=0)
        boot_importances.append(importance_b)

        rank_b = pd.Series(importance_b, index=feature_names).rank(ascending=False)
        rho, _ = spearmanr(ref_rank, rank_b)
        boot_rank_corrs.append(rho)

        focus_idx = list(feature_names).index(FOCUS_FEATURE) if FOCUS_FEATURE in feature_names else None
        if focus_idx is not None:
            focus_records.append({
                "Bootstrap": b,
                "Mean_abs_SHAP": importance_b[focus_idx],
                "Rank": rank_b.iloc[focus_idx],
            })

        if (b + 1) % 5 == 0:
            print(f"  Bootstrap {b + 1}/{n_boot} done (Spearman rho vs. reference = {rho:.3f})")

    boot_importances = np.array(boot_importances)  # shape (n_boot, n_features)

    stability_df = pd.DataFrame({
        "Feature": feature_names,
        "Reference_Mean_abs_SHAP": ref_importance,
        "Reference_Rank": ref_rank.values,
        "Bootstrap_Mean_abs_SHAP": boot_importances.mean(axis=0),
        "Bootstrap_Std_abs_SHAP": boot_importances.std(axis=0),
        "Bootstrap_CV": boot_importances.std(axis=0) / (boot_importances.mean(axis=0) + 1e-12),
    })
    stability_df = stability_df.sort_values("Reference_Rank")
    stability_df.to_csv(os.path.join(OUT_DIR, "shap_bootstrap_rank_stability.csv"), index=False)

    print(f"\nOverall SHAP ranking stability across {n_boot} bootstraps:")
    print(f"  Mean Spearman rho (bootstrap ranking vs. reference ranking): "
          f"{np.mean(boot_rank_corrs):.3f}  (std: {np.std(boot_rank_corrs):.3f})")

    focus_df = pd.DataFrame(focus_records)

    return stability_df, boot_rank_corrs, focus_df, ref_rf, ref_shap, ref_importance


In [7]:
def cluster_level_stability(cluster_df, feature_names, ref_importance, boot_importances_matrix=None,
                             stability_df=None):
    # Recompute cluster totals from the bootstrap stability table we already saved,
    # using per-feature bootstrap mean|SHAP| as a proxy (avoids recomputing SHAP again).
    merged = stability_df.merge(cluster_df, on="Feature")
    cluster_summary = merged.groupby("Cluster").agg(
        Features=("Feature", lambda s: list(s)),
        N_Features=("Feature", "count"),
        Reference_Cluster_SHAP=("Reference_Mean_abs_SHAP", "sum"),
        Bootstrap_Cluster_SHAP_Mean=("Bootstrap_Mean_abs_SHAP", "sum"),
        Max_Feature_Rank_Range=("Reference_Rank", lambda s: s.max() - s.min()),
    ).reset_index()
    cluster_summary = cluster_summary[cluster_summary.N_Features > 1].sort_values(
        "Reference_Cluster_SHAP", ascending=False
    )
    cluster_summary.to_csv(os.path.join(OUT_DIR, "shap_cluster_stability.csv"), index=False)
    print("\nCluster-level (grouped) SHAP importance for correlated feature groups:")
    print(cluster_summary.to_string(index=False))
    return cluster_summary


In [8]:
def background_sensitivity(ref_rf, X_train_s, X_eval_s, feature_names):
    rng = np.random.RandomState(RANDOM_STATE)
    records = []

    background_variants = {
        "path_dependent (as in original notebooks)": None,  # uses TreeExplainer default, no explicit background
        "interventional_100_sample": X_train_s[rng.choice(len(X_train_s), size=100, replace=False)],
        "interventional_full_train": X_train_s,
    }

    per_variant_importance = {}
    for name, background in background_variants.items():
        if background is None:
            explainer = shap.TreeExplainer(ref_rf, feature_perturbation="tree_path_dependent")
        else:
            explainer = shap.TreeExplainer(ref_rf, data=background, feature_perturbation="interventional")
        shap_vals = positive_class_shap(explainer.shap_values(X_eval_s, check_additivity=False))
        importance = np.abs(shap_vals).mean(axis=0)
        per_variant_importance[name] = importance

        focus_idx = list(feature_names).index(FOCUS_FEATURE) if FOCUS_FEATURE in feature_names else None
        rank_series = pd.Series(importance, index=feature_names).rank(ascending=False)

        records.append({
            "Background_variant": name,
            "Top10_features": ", ".join(rank_series.sort_values().head(10).index.tolist()),
            f"{FOCUS_FEATURE}_mean_abs_SHAP": importance[focus_idx] if focus_idx is not None else np.nan,
            f"{FOCUS_FEATURE}_rank": rank_series.iloc[focus_idx] if focus_idx is not None else np.nan,
        })

    bg_df = pd.DataFrame(records)
    bg_df.to_csv(os.path.join(OUT_DIR, "shap_background_sensitivity.csv"), index=False)
    print("\nBackground-data sensitivity of SHAP rankings:")
    print(bg_df.to_string(index=False))

    # Pairwise Spearman rho between variants' full importance vectors
    names = list(per_variant_importance.keys())
    print("\nPairwise Spearman rho of mean|SHAP| importance vectors across background choices:")
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            rho, _ = spearmanr(per_variant_importance[names[i]], per_variant_importance[names[j]])
            print(f"  {names[i]}  vs.  {names[j]}: rho = {rho:.3f}")

    return bg_df




In [9]:
def main():
    X, y = load_data()
    feature_names = X.columns.tolist()

    X_train, X_eval, y_train, y_eval = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_eval_s = scaler.transform(X_eval)

    # Fixed, smaller evaluation sample so SHAP recomputation across ~30 bootstraps is tractable
    rng = np.random.RandomState(RANDOM_STATE)
    eval_idx = rng.choice(len(X_eval_s), size=min(EVAL_SAMPLE_SIZE, len(X_eval_s)), replace=False)
    X_eval_fixed = X_eval_s[eval_idx]

    print("Step 1: correlated-feature clustering")
    cluster_df = cluster_correlated_features(X, threshold=CORR_THRESHOLD)

    print("\nStep 2: bootstrap resampling SHAP stability (this refits Random Forest %d times)" % N_BOOTSTRAPS)
    stability_df, boot_rank_corrs, focus_df, ref_rf, ref_shap, ref_importance = bootstrap_shap_stability(
        X_train_s, y_train, X_eval_fixed, feature_names, n_boot=N_BOOTSTRAPS
    )

    print("\nStep 2b: cluster-level (grouped) importance stability")
    cluster_summary = cluster_level_stability(cluster_df, feature_names, ref_importance, stability_df=stability_df)

    print("\nStep 3: background-data sensitivity")
    bg_df = background_sensitivity(ref_rf, X_train_s, X_eval_fixed, feature_names)

    # ---- Written summary ----
    lines = []
    lines.append("SHAP stability analysis summary")
    lines.append("=" * 40)
    lines.append(f"Bootstrap resamples: {N_BOOTSTRAPS} | Fixed evaluation sample size: {len(X_eval_fixed)}")
    lines.append(f"Mean Spearman rank correlation (bootstrap vs. reference ranking): "
                 f"{np.mean(boot_rank_corrs):.3f} (std {np.std(boot_rank_corrs):.3f})")
    if not focus_df.empty:
        lines.append(f"\n{FOCUS_FEATURE} across bootstraps: "
                     f"mean rank {focus_df.Rank.mean():.1f} (std {focus_df.Rank.std():.1f}), "
                     f"mean|SHAP| {focus_df.Mean_abs_SHAP.mean():.4f} (std {focus_df.Mean_abs_SHAP.std():.4f})")
    lines.append(
        "\nCorrelated feature clusters were detected among the page-count / duration / rate "
        "variables (see feature_correlation_clusters.csv). Where an individual feature's rank "
        "varies substantially across bootstraps (high Bootstrap_CV in "
        "shap_bootstrap_rank_stability.csv) but its cluster's combined attribution "
        "(shap_cluster_stability.csv) stays comparatively stable, this indicates SHAP mass is "
        "being redistributed among correlated proxies rather than reflecting genuine uncertainty "
        "about the underlying behavioural construct — a point the manuscript should make "
        "explicit rather than reporting single-feature ranks as though they were independent."
    )
    lines.append(
        "\nBackground-data sensitivity results (shap_background_sensitivity.csv) show whether the "
        "top-10 ranking and the Month_Nov attribution are an artifact of the path-dependent "
        "TreeExplainer default used in the original notebooks, or whether they replicate under "
        "interventional SHAP with different background references."
    )

    with open(os.path.join(OUT_DIR, "shap_stability_summary.txt"), "w") as f:
        f.write("\n".join(lines))
    print("\n" + "\n".join(lines))
    print(f"\nSaved all outputs under {OUT_DIR}/")




In [ ]:
if __name__ == "__main__":
    main()


Step 1: correlated-feature clustering
Correlated feature clusters (|Spearman rho| >= 0.70):
  Cluster 1: ['OperatingSystems_5', 'Browser_11']
  Cluster 13: ['Browser_13', 'OperatingSystems_8']
  Cluster 19: ['Informational_Duration', 'Informational']
  Cluster 20: ['Administrative_Duration', 'Administrative']
  Cluster 21: ['ProductRelated', 'ProductRelated_Duration']

Step 2: bootstrap resampling SHAP stability (this refits Random Forest 30 times)
